# GCI In-Class Competition: Home Credit Default Risk
## Notebook 2: Optuna Hyperparameter Optimization

**Objective**: Find optimal hyperparameters for LightGBM, XGBoost, CatBoost
**Method**: Bayesian optimization with 3-fold CV, 50 trials each
**Hardware**: Google Colab T4 GPU
**Prerequisite**: Run Notebook 1 (01_feature_engineering.ipynb) first

---

### Optimization Strategy
- **LightGBM**: GPU accelerated (`device='gpu'`), 50 trials
- **XGBoost**: GPU accelerated (`tree_method='gpu_hist'`), 50 trials  
- **CatBoost**: GPU accelerated (`task_type='GPU'`), 50 trials
- **CV**: 3-fold StratifiedKFold (faster than 5-fold for search)
- **Metric**: Out-of-fold ROC-AUC
- **Pruner**: MedianPruner for early stopping of bad trials
- **Fixed**: `scale_pos_weight=11.4` (class imbalance)

⚠️ **Time estimate**: ~2-3 hours total on T4 GPU

In [ ]:
# 1. Setup and Library Imports
import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
import catboost as cb
import optuna
from optuna.pruners import MedianPruner
from optuna.samplers import TPESampler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import time
import os
import gc
import json
import warnings
warnings.filterwarnings('ignore')

print(f"Optuna: {optuna.__version__}")
print(f"LightGBM: {lgb.__version__}")
print(f"XGBoost: {xgb.__version__}")
print(f"CatBoost: {cb.__version__}")

In [ ]:
# 2. Mount Drive & Load Engineered Data
from google.colab import drive
drive.mount('/content/drive')

WORK_DIR = '/content/drive/MyDrive/gci_competition'

# Load engineered datasets
train = pd.read_parquet(f'{WORK_DIR}/features/train_engineered.parquet')
test = pd.read_parquet(f'{WORK_DIR}/features/test_engineered.parquet')

# Load metadata
with open(f'{WORK_DIR}/features/feature_cols.json', 'r') as f:
    feature_cols = json.load(f)

y = np.load(f'{WORK_DIR}/features/y_train.npy')

with open(f'{WORK_DIR}/features/high_card_cols.json', 'r') as f:
    high_card_cols = json.load(f)

with open(f'{WORK_DIR}/features/high_corr_features.json', 'r') as f:
    high_corr_features = json.load(f)

# Prepare X
X = train[feature_cols].copy()
X_test = test[feature_cols].copy()

# Fill NaN
X = X.fillna(-999)
X_test = X_test.fillna(-999)

print(f"Train shape: {X.shape}")
print(f"Test shape: {X_test.shape}")
print(f"Features: {len(feature_cols)}")
print(f"Target shape: {y.shape}")
print(f"Positive rate: {y.mean():.4f}")
print(f"Scale_pos_weight: {(1-y.mean())/y.mean():.2f}")

In [ ]:
# 3. Identify Categorical Feature Indices for Each Model

# For LightGBM: categorical_feature parameter accepts column names or indices
# For CatBoost: cat_features parameter accepts column indices
# For XGBoost: enable_categorical=True + categorical indices

# Get categorical columns (from label encoding + frequency encoding)
cat_cols_lgb = []
for col in X.columns:
    if X[col].dtype in ['int32', 'int64', 'int8', 'int16']:
        # Check if it's a label-encoded categorical (low cardinality)
        if X[col].nunique() < 100:
            cat_cols_lgb.append(col)
    elif X[col].dtype == 'category':
        cat_cols_lgb.append(col)

# Also include original categorical column names
original_cats = [
    'NAME_CONTRACT_TYPE', 'CODE_GENDER', 'NAME_INCOME_TYPE',
    'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE',
    'OCCUPATION_TYPE', 'ORGANIZATION_TYPE', 'FLAG_MOBIL',
    'FLAG_EMP_PHONE', 'FLAG_WORK_PHONE'
]

# Find indices
cat_indices_lgb = [feature_cols.index(c) for c in original_cats if c in feature_cols]
cat_indices_cat = cat_indices_lgb.copy()  # Same for CatBoost
cat_indices_xgb = cat_indices_lgb.copy()  # Same for XGBoost

print(f"Categorical feature indices for LightGBM: {len(cat_indices_lgb)}")
print(f"Categorical feature indices for CatBoost: {len(cat_indices_cat)}")
print(f"Categorical feature indices for XGBoost: {len(cat_indices_xgb)}")

# Verify
for idx in cat_indices_lgb:
    print(f"  [{idx}] {feature_cols[idx]}: {X.iloc[:, idx].nunique()} unique")

In [ ]:
# 4. LightGBM Objective Function for Optuna

SCALE_POS_WEIGHT = (1 - y.mean()) / y.mean()  # ~11.4
N_SPLITS = 3
RANDOM_STATE = 42
EARLY_STOPPING_ROUNDS = 50
N_ESTIMATORS = 2000

def lgb_objective(trial):
    """Optuna objective for LightGBM."""
    
    # Hyperparameter search space
    params = {
        'objective': 'binary',
        'metric': 'auc',
        'boosting_type': 'gbdt',
        'device': 'gpu',
        'gpu_platform_id': 0,
        'gpu_device_id': 0,
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 16, 256),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        'min_split_gain': trial.suggest_float('min_split_gain', 0.0, 1.0),
        'min_child_weight': trial.suggest_float('min_child_weight', 1e-3, 10.0, log=True),
        'scale_pos_weight': SCALE_POS_WEIGHT,
        'random_state': RANDOM_STATE,
        'n_jobs': 2,
        'verbose': -1
    }
    
    # CV
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    oof_preds = np.zeros(len(X))
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        X_tr, y_tr = X.iloc[train_idx], y[train_idx]
        X_va, y_va = X.iloc[val_idx], y[val_idx]
        
        # Apply target encoding for high-cardinality categoricals WITHIN FOLD
        # (This is a simplified version - full implementation in notebook 3)
        
        model = lgb.LGBMClassifier(**params, n_estimators=N_ESTIMATORS)
        
        callbacks = [
            lgb.early_stopping(stopping_rounds=EARLY_STOPPING_ROUNDS, verbose=False),
            lgb.log_evaluation(period=0)
        ]
        
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_va, y_va)],
            eval_metric='auc',
            categorical_feature=cat_indices_lgb if cat_indices_lgb else 'auto',
            callbacks=callbacks
        )
        
        val_pred = model.predict_proba(X_va)[:, 1]
        oof_preds[val_idx] = val_pred
    
    cv_score = roc_auc_score(y, oof_preds)
    
    # Report intermediate value for pruning
    trial.report(cv_score, step=0)
    
    return cv_score

print("LightGBM objective function defined!")

In [ ]:
# 5. XGBoost Objective Function for Optuna

def xgb_objective(trial):
    """Optuna objective for XGBoost."""
    
    params = {
        'objective': 'binary:logistic',
        'eval_metric': 'auc',
        'tree_method': 'gpu_hist',
        'gpu_id': 0,
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'min_child_weight': trial.suggest_float('min_child_weight', 1e-3, 10.0, log=True),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'colsample_bylevel': trial.suggest_float('colsample_bylevel', 0.5, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        'gamma': trial.suggest_float('gamma', 0.0, 1.0),
        'scale_pos_weight': SCALE_POS_WEIGHT,
        'random_state': RANDOM_STATE,
        'n_jobs': 2,
        'verbosity': 0,
        'enable_categorical': True
    }
    
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    oof_preds = np.zeros(len(X))
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        X_tr, y_tr = X.iloc[train_idx], y[train_idx]
        X_va, y_va = X.iloc[val_idx], y[val_idx]
        
        # XGBoost requires DMatrix for categorical
        dtrain = xgb.DMatrix(X_tr, label=y_tr, enable_categorical=True)
        dval = xgb.DMatrix(X_va, label=y_va, enable_categorical=True)
        
        model = xgb.train(
            params,
            dtrain,
            num_boost_round=N_ESTIMATORS,
            evals=[(dval, 'val')],
            early_stopping_rounds=EARLY_STOPPING_ROUNDS,
            verbose_eval=False
        )
        
        val_pred = model.predict(dval, iteration_range=(0, model.best_iteration + 1))
        oof_preds[val_idx] = val_pred
    
    cv_score = roc_auc_score(y, oof_preds)
    trial.report(cv_score, step=0)
    
    return cv_score

print("XGBoost objective function defined!")

In [ ]:
# 6. CatBoost Objective Function for Optuna

def cat_objective(trial):
    """Optuna objective for CatBoost."""
    
    params = {
        'objective': 'Logloss',
        'eval_metric': 'AUC',
        'task_type': 'GPU',
        'devices': '0',
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'depth': trial.suggest_int('depth', 3, 10),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-3, 10.0, log=True),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 1.0),
        'random_strength': trial.suggest_float('random_strength', 1e-3, 10.0, log=True),
        'border_count': trial.suggest_int('border_count', 32, 255),
        'scale_pos_weight': SCALE_POS_WEIGHT,
        'random_seed': RANDOM_STATE,
        'verbose': False,
        'allow_writing_files': False
    }
    
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    oof_preds = np.zeros(len(X))
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        X_tr, y_tr = X.iloc[train_idx], y[train_idx]
        X_va, y_va = X.iloc[val_idx], y[val_idx]
        
        model = cb.CatBoostClassifier(**params, iterations=N_ESTIMATORS)
        
        model.fit(
            X_tr, y_tr,
            eval_set=(X_va, y_va),
            cat_features=cat_indices_cat if cat_indices_cat else None,
            early_stopping_rounds=EARLY_STOPPING_ROUNDS,
            verbose=False
        )
        
        val_pred = model.predict_proba(X_va)[:, 1]
        oof_preds[val_idx] = val_pred
    
    cv_score = roc_auc_score(y, oof_preds)
    trial.report(cv_score, step=0)
    
    return cv_score

print("CatBoost objective function defined!")

In [ ]:
# 7. Run Optuna Optimization for LightGBM (50 trials)

print("=" * 60)
print("Starting LightGBM Optuna Optimization (50 trials)")
print("=" * 60)

# Create study with TPE sampler and MedianPruner
sampler = TPESampler(seed=RANDOM_STATE)
pruner = MedianPruner(n_startup_trials=10, n_warmup_steps=5)

study_lgb = optuna.create_study(
    direction='maximize',
    sampler=sampler,
    pruner=pruner,
    study_name='lgb_tuning'
)

t0 = time.time()
study_lgb.optimize(lgb_objective, n_trials=50, timeout=7200, show_progress_bar=True)
elapsed = time.time() - t0

print(f"\nLightGBM Optimization Complete!")
print(f"Time: {elapsed/60:.1f} minutes")
print(f"Best AUC: {study_lgb.best_value:.5f}")
print(f"Best params: {study_lgb.best_params}")

# Save study
import joblib
joblib.dump(study_lgb, f'{WORK_DIR}/optuna_studies/study_lgb.pkl')

# Save best params
with open(f'{WORK_DIR}/optuna_studies/best_params_lgb.json', 'w') as f:
    json.dump(study_lgb.best_params, f, indent=2)

# Plot optimization history (optional)
try:
    optuna.visualization.plot_optimization_history(study_lgb).show()
    optuna.visualization.plot_param_importances(study_lgb).show()
except:
    pass

In [ ]:
# 8. Run Optuna Optimization for XGBoost (50 trials)

print("=" * 60)
print("Starting XGBoost Optuna Optimization (50 trials)")
print("=" * 60)

sampler = TPESampler(seed=RANDOM_STATE)
pruner = MedianPruner(n_startup_trials=10, n_warmup_steps=5)

study_xgb = optuna.create_study(
    direction='maximize',
    sampler=sampler,
    pruner=pruner,
    study_name='xgb_tuning'
)

t0 = time.time()
study_xgb.optimize(xgb_objective, n_trials=50, timeout=7200, show_progress_bar=True)
elapsed = time.time() - t0

print(f"\nXGBoost Optimization Complete!")
print(f"Time: {elapsed/60:.1f} minutes")
print(f"Best AUC: {study_xgb.best_value:.5f}")
print(f"Best params: {study_xgb.best_params}")

# Save study
joblib.dump(study_xgb, f'{WORK_DIR}/optuna_studies/study_xgb.pkl')

with open(f'{WORK_DIR}/optuna_studies/best_params_xgb.json', 'w') as f:
    json.dump(study_xgb.best_params, f, indent=2)

try:
    optuna.visualization.plot_optimization_history(study_xgb).show()
    optuna.visualization.plot_param_importances(study_xgb).show()
except:
    pass

In [ ]:
# 9. Run Optuna Optimization for CatBoost (50 trials)

print("=" * 60)
print("Starting CatBoost Optuna Optimization (50 trials)")
print("=" * 60)

sampler = TPESampler(seed=RANDOM_STATE)
pruner = MedianPruner(n_startup_trials=10, n_warmup_steps=5)

study_cat = optuna.create_study(
    direction='maximize',
    sampler=sampler,
    pruner=pruner,
    study_name='cat_tuning'
)

t0 = time.time()
study_cat.optimize(cat_objective, n_trials=50, timeout=7200, show_progress_bar=True)
elapsed = time.time() - t0

print(f"\nCatBoost Optimization Complete!")
print(f"Time: {elapsed/60:.1f} minutes")
print(f"Best AUC: {study_cat.best_value:.5f}")
print(f"Best params: {study_cat.best_params}")

# Save study
joblib.dump(study_cat, f'{WORK_DIR}/optuna_studies/study_cat.pkl')

with open(f'{WORK_DIR}/optuna_studies/best_params_cat.json', 'w') as f:
    json.dump(study_cat.best_params, f, indent=2)

try:
    optuna.visualization.plot_optimization_history(study_cat).show()
    optuna.visualization.plot_param_importances(study_cat).show()
except:
    pass

In [ ]:
# 10. Summary & Comparison

print("=" * 60)
print("OPTUNA OPTIMIZATION SUMMARY")
print("=" * 60)

# Load all best params
with open(f'{WORK_DIR}/optuna_studies/best_params_lgb.json', 'r') as f:
    best_lgb = json.load(f)
with open(f'{WORK_DIR}/optuna_studies/best_params_xgb.json', 'r') as f:
    best_xgb = json.load(f)
with open(f'{WORK_DIR}/optuna_studies/best_params_cat.json', 'r') as f:
    best_cat = json.load(f)

print(f"\nLightGBM:  AUC = {study_lgb.best_value:.5f}")
for k, v in best_lgb.items():
    print(f"  {k}: {v}")

print(f"\nXGBoost:   AUC = {study_xgb.best_value:.5f}")
for k, v in best_xgb.items():
    print(f"  {k}: {v}")

print(f"\nCatBoost:  AUC = {study_cat.best_value:.5f}")
for k, v in best_cat.items():
    print(f"  {k}: {v}")

# Save combined best params
all_best = {
    'lightgbm': best_lgb,
    'xgboost': best_xgb,
    'catboost': best_cat
}
with open(f'{WORK_DIR}/optuna_studies/all_best_params.json', 'w') as f:
    json.dump(all_best, f, indent=2)

print("\n✅ Notebook 2 Complete! Ready for Notebook 3: Ensemble Training")